# 12 - Cost-Aware Multi-Objective qLogEHVI Campaign

This notebook demonstrates the v1.1.3 backend workflow for coupled multi-objective qLogEHVI with deterministic cost ranking. Cost is used for batch utility reranking; it is not modeled as an additional objective.

In [1]:
from pathlib import Path
from shutil import copyfile
import os
import sys

import pandas as pd

from bo_forge.session import CampaignSession

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

mpl_cache = PROJECT_ROOT / ".matplotlib-cache"
mpl_cache.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_cache))

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs" / "12_cost_aware_multi_objective_qlogehvi.yaml"
SEED_LOG_PATH = PROJECT_ROOT / "examples" / "12_cost_aware_multi_objective_campaign_log.csv"
WORKING_LOG_PATH = PROJECT_ROOT / "examples" / "12_cost_aware_multi_objective_working_log.csv"
LATEST_SUGGESTIONS_PATH = PROJECT_ROOT / "examples" / "12_cost_aware_multi_objective_latest_suggestions.csv"
REPORT_PATH = PROJECT_ROOT / "reports" / "12_cost_aware_multi_objective_report.txt"
COST_PROGRESS_PATH = PROJECT_ROOT / "reports" / "12_cost_aware_multi_objective_cost_progress.png"
PARETO_PATH = PROJECT_ROOT / "reports" / "12_cost_aware_multi_objective_pareto.png"
TARGET_OBSERVED_ROWS = 15

Copy the seed log to an ignored working log so the committed example remains unchanged.

In [2]:
copyfile(SEED_LOG_PATH, WORKING_LOG_PATH)
campaign = CampaignSession.from_files(CONFIG_PATH, WORKING_LOG_PATH)
campaign.validate()
campaign.summary()

,field,value
0,campaign_name,cost_aware_multi_objective_tradeoff
1,campaign_status,ready_for_bo
2,objectives,"yield, selectivity, waste, energy_use"
3,directions,"maximize, maximize, minimize, minimize"
4,reference_points,"yield=0.2, selectivity=0.2, waste=0.9, energy_..."
5,total_rows,5
6,observed_rows,5
7,pending_suggestions,0
8,initial_design_remaining,0
9,next_iteration,5


The synthetic evaluator returns all objectives together because BO Forge currently assumes coupled multi-objective observations.

In [3]:
def simulate(row: pd.Series) -> dict[str, float]:
    loading = float(row["catalyst_loading"])
    time = float(row["reaction_time"])
    base = float(row["base_equivalents"])
    solvent = str(row["solvent"])
    solvent_yield = {"MeCN": 0.13, "DMF": 0.02, "Water": -0.08}[solvent]
    solvent_selectivity = {"MeCN": -0.05, "DMF": 0.02, "Water": 0.16}[solvent]
    yield_value = 0.25 + 3.8 * loading + 0.003 * time - 0.08 * base + solvent_yield
    selectivity = 0.35 + 0.004 * time - 1.0 * loading + solvent_selectivity
    waste = 0.72 - 1.8 * loading + 0.06 * base + (0.12 if solvent == "DMF" else -0.10 if solvent == "Water" else 0.0)
    energy = 0.12 + 0.0065 * time + 0.04 * base + (0.04 if solvent == "Water" else 0.0)
    return {
        "yield": max(0.0, min(1.0, yield_value)),
        "selectivity": max(0.0, min(1.0, selectivity)),
        "waste": max(0.0, min(1.0, waste)),
        "energy_use": max(0.0, min(1.0, energy)),
    }

Generate cost-aware qLogEHVI suggestions, accept them when review is enabled, mark coupled observations, and stop at the target count.

In [4]:
records = []
while len(campaign.observed_data()) < TARGET_OBSERVED_ROWS:
    suggestions = campaign.suggest_next(batch_size=2)
    suggestions.to_csv(LATEST_SUGGESTIONS_PATH, index=False)
    campaign.append_suggestions(suggestions)
    for row_id in suggestions["row_id"]:
        campaign.review_suggestion(str(row_id), "accept", note="notebook demo")
    for _, row in campaign.pending_suggestions().iterrows():
        if str(row.get("review_status", "")) != "accepted":
            continue
        values = simulate(row)
        actual_cost = float(row["cost_estimate"]) * 1.02
        campaign.mark_observed(str(row["row_id"]), objective_values=values, actual_cost=actual_cost)
        records.append({"row_id": row["row_id"], **values, "actual_cost": actual_cost})
        if len(campaign.observed_data()) >= TARGET_OBSERVED_ROWS:
            break

pd.DataFrame(records).tail()

SuggestionError: Could not generate a budget-feasible cost-aware qLogEHVI batch after 8 retries. remaining_budget=0.3254. Last rejection: batch exceeds remaining budget: total_batch_cost=8.46, remaining_budget=0.3254, candidates=[(0.08176185905933381, 55, 1.0, 'Water'), (0.12741419911384583, 68, 1.0, 'Water')].

Inspect the final cost, Pareto, and hypervolume state.

In [ ]:
campaign.summary(), campaign.cost_summary(), campaign.pareto_summary()

In [ ]:
campaign.export_report(REPORT_PATH)
campaign.plot_cost_progress(save_path=COST_PROGRESS_PATH)
campaign.plot_pareto(save_path=PARETO_PATH)